# Uczenie nadzorowane - klasyfikacja

## Wstęp

Projekt dotyczy problemu automatycznej klasyfikacji win na podstawie ich składu fizykochemicznego. Zbiór danych pochodzi z analizy chemicznej win uprawianych w tym samym regionie we Włoszech, ale wyprodukowanych z trzech różnych odmian winorośli (kultywarów).


W ramach badania przeanalizowano 178 próbek, z których każda opisana jest przez 13 atrybutów ciągłych, takich jak zawartość alkoholu, kwasu jabłkowego, popiołu, magnezu, fenoli czy flawonoidów. Dane te charakteryzują się dobrze zdefiniowaną strukturą klas, co czyni je doskonałym materiałem do testowania sprawności algorytmów uczenia maszynowego w zadaniach klasyfikacji wieloklasowej.

## Cel projektu

Głównym celem jest zbudowanie modeli uczenia maszynowego zdolnych do bezbłędnej klasyfikacji odmiany wina. W projekcie porównujemy:

**K-Nearest Neighbors (k-NN)** – model geometryczny bazujący na odległości.

**Random Forest** – model zespołowy oparty na drzewach decyzyjnych.

Kluczowym elementem jest optymalizacja hiperparametrów za pomocą GridSearchCV oraz automatyzacja procesu przy użyciu Pipeline.

<br>

Szczegółowe cele obejmują:  
Przygotowanie danych: Opracowanie potoku przetwarzania (Pipeline), który obejmuje standaryzację zmiennych, co jest kluczowe dla algorytmów wrażliwych na skalę atrybutów, takich jak k-NN.

Optymalizację modeli: Wykorzystanie metody przeszukiwania siatki (GridSearchCV) w celu automatycznego doboru optymalnych hiperparametrów dla obu klasyfikatorów.

Ewaluację porównawczą: Ocena skuteczności modeli przy użyciu zaawansowanych metryk, takich jak dokładność (accuracy), precyzja (precision), czułość (recall) oraz analiza macierzy pomyłek (confusion matrix).


Weryfikację separowalności: Sprawdzenie w praktyce tezy o wysokiej separowalności klas w tym zbiorze danych poprzez analizę błędów popełnianych przez poszczególne algorytmy.

# 1. Wczytanie danych z pliku

In [25]:
# Wymagane biblioteki
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

columns = [
    'Class', 'Alcohol', 'Malic_acid', 'Ash', 'Alcalinity_of_ash',
    'Magnesium', 'Total_phenols', 'Flavanoids', 'Nonflavanoid_phenols',
    'Proanthocyanins', 'Color_intensity', 'Hue', 'OD280_OD315', 'Proline'
]

df = pd.read_csv('wine.data', names=columns, header=None)

df.head()


,Class,Alcohol,Malic_acid,Ash,Alcalinity_of_ash,Magnesium,Total_phenols,Flavanoids,Nonflavanoid_phenols,Proanthocyanins,Color_intensity,Hue,OD280_OD315,Proline
0,1,14.23,1.71,2.43,15.6,127,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065
1,1,13.20,1.78,2.14,11.2,100,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050
2,1,13.16,2.36,2.67,18.6,101,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185
3,1,14.37,1.95,2.50,16.8,113,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480
4,1,13.24,2.59,2.87,21.0,118,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735


# 2. Podział na zbiór treningowy i testowy

In [26]:
# Podział na zmienne niezależne (X - klasyfikatory) i zależną (y - target)
X = df.drop('Class', axis=1) # Macierz cech (13 atrybutów)
y = df['Class']              # Wektor typu wina (1, 2, 3)

# Podział na zbiór treningowy i testowy
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# 3. Normalizacja danych

In [27]:
# Lista cech numerycznych (wszystkie 13 atrybutów )
num_features = X.columns.tolist()

# Definicja preprocesora (tylko normalizacja)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features)
    ]
)

Po co jest **normalizacja** i jak może ona wpływać na działania algorytmów kNN i random forest?  

Cel normalizacji:
- sprowadza wszystkie cechy do wspólnej skali (przekształca dane tak, aby każda cecha miała średnią 0 i odchylenie standardowe 1),
- zapobiega dominowaniu cech o dużych wartościach liczbowych.  

Wpływ na algorytmy:
- kNN: kluczowy wpływ; bo algorytm opiera się na odległościach między punktami (bez skalowania, cecha o dużej rozpiętości (np. Proline, zakres do 1680) zdominowałaby cechy o małych wartościach (np. Nonflavanoid phenols, zakres < 1), czyniąc je nieistotnymi dla modelu),
- Random Forest: brak wpływu, ten algorytm jest odporny na skalowanie. Drzewa decyzyjne dzielą dane na podstawie progów, więc przeskalowanie wartości nie zmienia struktury podziałów (drzewa decyzyjne operują na progach wartości pojedynczych cech, a nie na ich wzajemnych odległościach).



#

# 4. Definicja Pipeline'ów


In [28]:
# Trening algorytmów

# Definicja Pipeline'ów dla obu modeli
pipeline_knn = Pipeline([
    ('prep', preprocessor),
    ('clf', KNeighborsClassifier())
])

pipeline_rf = Pipeline([
    ('prep', preprocessor),
    ('clf', RandomForestClassifier(random_state=42))
])

# 5a. Optymalizacja hiperparametrów (GridSearchCV)

In [29]:
# Siatki parametrów do przeszukania
knn_params = {
    'clf__n_neighbors': [3, 5, 7, 11],
    'clf__weights': ['uniform', 'distance'],
    'clf__metric': ['euclidean', 'manhattan']
}

rf_params = {
    'clf__n_estimators': [50, 100, 200],
    'clf__max_depth': [None, 5, 10],
    'clf__min_samples_split': [2, 5]
}

# GridSearchCV
knn_search = GridSearchCV(pipeline_knn, knn_params, cv=5).fit(X_train, y_train)
rf_search = GridSearchCV(pipeline_rf, rf_params, cv=5).fit(X_train, y_train)

# Wyniki
print(f"Najlepsze parametry KNN: {knn_search.best_params_}")
print(f"Najlepsze parametry RF: {rf_search.best_params_}")

Najlepsze parametry KNN: {'clf__metric': 'manhattan', 'clf__n_neighbors': 5, 'clf__weights': 'uniform'}
Najlepsze parametry RF: {'clf__max_depth': None, 'clf__min_samples_split': 2, 'clf__n_estimators': 200}


# 5. Trening algorytmów

In [8]:
# pipeline_knn.fit(X_train, y_train)
# pipeline_rf.fit(X_train, y_train)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Alcohol', 'Malic_acid',
                                                   'Ash', 'Alcalinity_of_ash',
                                                   'Magnesium', 'Total_phenols',
                                                   'Flavanoids',
                                                   'Nonflavanoid_phenols',
                                                   'Proanthocyanins',
                                                   'Color_intensity', 'Hue',
                                                   'OD280_OD315',
                                                   'Proline'])])),
                ('clf', RandomForestClassifier(random_state=42))])

# 6. Klasyfikacja

In [30]:
# y_pred_knn = pipeline_knn.predict(X_test)
# y_pred_rf = pipeline_rf.predict(X_test)

y_pred_knn = knn_search.predict(X_test)
y_pred_rf = rf_search.predict(X_test)

# 7. Raport z klasyfikacji

In [33]:
# Wyświetlenie Accuracy, raportów i macierzy pomyłek
print("KNN accuracy:", accuracy_score(y_test, y_pred_knn))
print("\n--- KNN Classification Report ---")
print(classification_report(y_test, y_pred_knn))
print("\nKNN confusion matrix:\n", confusion_matrix(y_test, y_pred_knn))

print("\nRandom Forest accuracy:", accuracy_score(y_test, y_pred_rf))
print("\n--- Random Forest Classification Report ---")
print(classification_report(y_test, y_pred_rf))
print("\nRandom Forest confusion matrix:\n", confusion_matrix(y_test, y_pred_rf))



KNN accuracy: 0.9814814814814815

--- KNN Classification Report ---
              precision    recall  f1-score   support

           1       1.00      1.00      1.00        18
           2       1.00      0.95      0.98        21
           3       0.94      1.00      0.97        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54


KNN confusion matrix:
 [[18  0  0]
 [ 0 20  1]
 [ 0  0 15]]

Random Forest accuracy: 1.0

--- Random Forest Classification Report ---
              precision    recall  f1-score   support

           1       1.00      1.00      1.00        18
           2       1.00      1.00      1.00        21
           3       1.00      1.00      1.00        15

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54


Random Forest confusion matrix:
 [

# Metryki

Metryki klasyfikacji:  
- **Accuracy (Dokładność)** - Odsetek wszystkich poprawnych decyzji modelu. Używamy jej, gdy klasy w zbiorze są zrównoważone (TP+TN)/(TP+TN+FP+FN)  

- **Precision (Precyzja)** - Mówi, ile z próbek zaklasyfikowanych jako dane wino faktycznie nim było. Zdolność modelu do nieoznaczania próbki ujemnej jako dodatniej. Chroni przed błędami typu False Positive. Mówi jak wiele z przewidzianych pozytywnych klas jest poprawnych (ważna, gdy fałszywe alarmy są kosztowne) TP/(TP+FP) (jak często jakaś klasa jest wybierana)

- **Recall (Czułość)** - Mówi, ile próbek danej odmiany wina model zdołał poprawnie wykryć z całej dostępnej puli tej odmiany. Zdolność modelu do znalezienia wszystkich próbek danej klasy. Mówi jak wiele rzeczywistych obiektów danej klasy zostało wykrytych (ważna, gdy pominięcie klasy jest kosztowne) TP/(TP+FN) (jak często jakaś klasa jest błędnie klasyfikowana)

- **F1-score** - Średnia harmoniczna precyzji i czułości. Jest lepszym miernikiem niż dokładność, gdy jedna z tych wartości jest niska. Najlepsza metryka, gdy szukamy balansu między "nie myleniem wina z innym" a "wykryciem wszystkich próbek" (2Precision*Recall)/(Precision+Recall)

- **Confusion Matrix (Macierz pomyłek)** - Macierz, która pokazuje, ile obiektów każdej klasy zostało sklasyfikowanych poprawnie, jakie są konkretne błędy klasyfikacji

- **Classification Report** - Zbiór powyższych metryk obliczony dla każdej klasy z osobna, dające pełny obraz jakości predykcji.





# Analiza metryk i predykcji poszczególnych modeli

Random Forest osiągnął 100% skuteczności na zbiorze treningowym, podczas gdy K-Neighbors Classifier (KNN) popełnił jeden błąd.  

Model KNN:
- Accuracy (Dokładność): Wynosi około 98,15\%, co oznacza, że na 54 próbki testowe, jedna została zaklasyfikowana błędnie.
- Confusion Matrix (Macierz pomyłek): Pokazuje dokładnie, gdzie nastąpił błąd. W drugim wierszu widzimy wartości [0, 20, 1]. Oznacza to, że jedna próbka, która faktycznie należała do Klasy 2, została błędnie zaklasyfikowana jako Klasa 3.
- Precision (Precyzja) dla Klasy 3: Wynosi 0.94. Wynika to z faktu, że model wskazał 16 próbek jako Klasa 3 (15 prawdziwych z Klasy 3 + 1 pomyłka z Klasy 2), więc jego "pewność" w tej klasie spadła.
- Recall (Czułość) dla Klasy 2: Wynosi 0.95 (20 poprawnych z 21 rzeczywistych). Model "przegapił" jedną próbkę tej klasy.
- F1-score: Nadal utrzymuje się na bardzo wysokim poziomie (0.98 dla Klasy 2 i 0.97 dla Klasy 3), co świadczy o ogólnej bardzo wysokiej sprawności modelu mimo drobnego potknięcia.  

Model Random Forest:
- Accuracy: 1.0
- Wszystkie metryki (Precision, Recall, F1): Wynoszą 1.00 dla każdej klasy.
- Interpretacja: Model ten bezbłędnie oddzielił wszystkie trzy typy win. Potwierdza to informację ze źródeł, że klasy w tym zbiorze danych są dobrze separowalne.

# Interpretacja różnic w działaniu modeli

Różnica w wynikach (100% vs 98.15%) wynika z faktu, że modele działają na zupełnie innych zasadach.


KNN opiera się na odległościach w 13-wymiarowej przestrzeni cech. Jeśli jedna próbka Klasy 2 posiada specyficzne stężenia składników chemicznych, które czynią ją "bliższą" geograficznie próbkom Klasy 3, KNN popełni błąd. Jest on bardzo wrażliwy na lokalne zagęszczenie danych.



Random Forest buduje zestaw drzew decyzyjnych, które szukają optymalnych punktów podziału dla każdej cechy z osobna. Potrafi on zignorować "szum" w jednej zmiennej, jeśli inne cechy wyraźnie wskazują na poprawną klasę. Jest to model bardziej odporny na outliery.


Wpływ cech:  
Zbiór zawiera 13 atrybutów ciągłych. Nie wszystkie są równie istotne.
- Random Forest automatycznie przeprowadza pewnego rodzaju selekcję cech, skupiając się na tych, które najlepiej różnicują klasy.  
- KNN bierze pod uwagę wszystkie cechy naraz. Nawet po standaryzacji, jeśli kilka mniej istotnych atrybutów w jednej próbce Klasy 2 upodobni ją do Klasy 3, algorytm "da się oszukać".

# Podsumowanie

Przeprowadzona analiza porównawcza algorytmów K-Nearest Neighbors oraz Random Forest na zbiorze danych Wine Recognition pozwoliła na sformułowanie następujących wniosków:

**Separowalność klas:** Dane chemiczne są bardzo silnie skorelowane z odmianą winorośli, co pozwala na niemal bezbłędną klasyfikację.

**Przewaga modelu zespołowego:** Random Forest okazał się stabilniejszy. Dzięki budowaniu wielu drzew decyzyjnych potrafi zignorować drobne anomalie w danych (szum), które "oszukują" algorytm k-NN bazujący na najbliższym sąsiedztwie.

Wrażliwość modelu KNN: Algorytm KNN popełnił jeden błąd (zaklasyfikowanie próbki z klasy 2 jako klasę 3), co uwidoczniło jego specyfikę – opieranie się na geometrycznej bliskości punktów w przestrzeni cech3. Nawet przy zastosowaniu optymalizacji GridSearchCV, jedna z próbek chemicznych odmiany 2 okazała się statystycznie zbyt podobna do odmiany 3, co zmyliło algorytm.

Kluczowa rola preprocesingu: Sukces obu modeli, a w szczególności wysoki wynik KNN (0.98), był możliwy dzięki zastosowaniu standaryzacji zmiennych. Ponieważ atrybuty takie jak Proline mają wartości mierzone w tysiącach, a Nonflavanoid phenols w ułamkach, sprowadzenie ich do wspólnej skali było niezbędne dla rzetelnego działania metryk odległości.

Wartość automatyzacji: Zastosowanie potoku Pipeline wraz z GridSearchCV pozwoliło na obiektywny wybór parametrów (np. metryki Manhattan dla KNN), co zminimalizowało ryzyko błędu ludzkiego przy ręcznym strojeniu modelu.

**Rekomendacja:** Do automatycznego rozpoznawania win na podstawie składu chemicznego zaleca się model Random Forest, ze względu na jego odporność na specyficzne odchylenia w pojedynczych pomiarach.